In [ ]:
!pip install metrics
!pip install segmentation_models_pytorch

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.utils as vutils
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset, Dataset


import cv2

from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import torch.nn as nn

import gc

from datasets import load_dataset


In [ ]:
import numpy as np
import torch

class StreamingMetrics:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.intersection = np.zeros(self.num_classes)
        self.union = np.zeros(self.num_classes)
    """
    def update(self, preds, targets):
        preds = preds.cpu().numpy()
        targets = targets.cpu().numpy()
        # Move to CPU and convert to numpy
        if torch.is_tensor(preds):
            preds = preds.detach().cpu().numpy()
        if torch.is_tensor(targets):
            targets = targets.detach().cpu().numpy()

        preds = preds.flatten()
        targets = targets.flatten()

        for cls in range(self.num_classes):
            pred_inds = (preds == cls)
            target_inds = (targets == cls)

            inter = np.logical_and(pred_inds, target_inds).sum()
            union = np.logical_or(pred_inds, target_inds).sum()

            self.intersection[cls] += inter
            self.union[cls] += union
    """
    def update(self, preds, targets):
        preds = preds.view(-1)
        targets = targets.view(-1)
    
        for cls in range(self.num_classes):
            pred_inds = (preds == cls)
            target_inds = (targets == cls)
    
            inter = (pred_inds & target_inds).sum().item()
            union = (pred_inds | target_inds).sum().item()
    
            self.intersection[cls] += inter
            self.union[cls] += union

    def compute(self):
        """
        Calculates and returns all metrics in a dictionary.
        This fixes the AttributeError.
        """
        # Intersection over Union
        iou_per_class = self.intersection / (self.union + 1e-8)
        mean_iou = np.mean(iou_per_class)

        # Dice Coefficient
        # Note: (P + T) = Intersection + Union
        dice_per_class = (2 * self.intersection) / (self.intersection + self.union + 1e-8)
        mean_dice = np.mean(dice_per_class)

        return {
            "mean_iou": mean_iou,
            "mean_dice": mean_dice,
            "iou_per_class": iou_per_class,
            "dice_per_class": dice_per_class
        }

In [ ]:
"""
import numpy as np

class StreamingMetrics:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.reset()

    def reset(self):
        self.intersection = np.zeros(self.num_classes)
        self.union = np.zeros(self.num_classes)

    def update(self, preds, targets):
        preds = preds.flatten()
        targets = targets.flatten()

        for cls in range(self.num_classes):
            pred_inds = preds == cls
            target_inds = targets == cls

            inter = np.logical_and(pred_inds, target_inds).sum()
            union = np.logical_or(pred_inds, target_inds).sum()

            self.intersection[cls] += inter
            self.union[cls] += union

    def get_mean_iou(self):
        iou = self.intersection / (self.union + 1e-8)
        return np.mean(iou)

"""

In [ ]:
import sys

from datasets import load_dataset
sys.path.append('/kaggle/input/datasets/iristhomas2507/utility')
import utils as u

In [ ]:
print(os.path.exists('/kaggle/input/datasets/iristhomas2507/unet-semantic-models/best_model_fold_0.pth'))

In [ ]:
dataset = load_dataset("RationAI/PanNuke")

print(dataset)

In [ ]:
path0 = "/kaggle/input/datasets/iristhomas2507/unet-semantic-models/best_model_fold0.pth" 
path1 = "/kaggle/input/datasets/iristhomas2507/unet-semantic-models/best_model_fold1.pth" 
path2= "/kaggle/input/datasets/iristhomas2507/unet-semantic-models/best_model_fold2.pth"

In [ ]:


def evaluate_model(model, dataloader, device, num_classes):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)

            outputs = model(images)  # (B, C, H, W)
            preds = torch.argmax(outputs, dim=1)  # (B, H, W)

            all_preds.append(preds.cpu())
            all_targets.append(masks)

    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    return all_preds, all_targets

In [ ]:
def compute_dice(pred, target, num_classes, eps=1e-6):
    dices = []

    for c in range(1, num_classes):  # ignore background
        pred_c = (pred == c)
        target_c = (target == c)

        intersection = (pred_c & target_c).sum().float()
        union = pred_c.sum() + target_c.sum()

        dice = (2 * intersection + eps) / (union + eps)
        dices.append(dice.item())

    return dices, np.mean(dices)

In [ ]:
def compute_iou(pred, target, num_classes, eps=1e-6):
    ious = []

    for c in range(1, num_classes):
        pred_c = (pred == c)
        target_c = (target == c)

        intersection = (pred_c & target_c).sum().float()
        union = (pred_c | target_c).sum().float()

        iou = (intersection + eps) / (union + eps)
        ious.append(iou.item())

    return ious, np.mean(ious)

In [ ]:
def compute_precision_recall(pred, target, num_classes, eps=1e-6):
    precision = []
    recall = []

    for c in range(1, num_classes):
        pred_c = (pred == c)
        target_c = (target == c)

        tp = (pred_c & target_c).sum().float()
        fp = (pred_c & ~target_c).sum().float()
        fn = (~pred_c & target_c).sum().float()

        prec = (tp + eps) / (tp + fp + eps)
        rec = (tp + eps) / (tp + fn + eps)

        precision.append(prec.item())
        recall.append(rec.item())

    return precision, recall

In [ ]:
def compute_accuracy(pred, target):
    return (pred == target).float().mean().item()

In [ ]:
import matplotlib.pyplot as plt

def visualize_samples(images, masks, preds, num_samples=20):
    indices = np.random.choice(len(images), num_samples, replace=False)

    for i in indices:
        img = images[i].permute(1,2,0).numpy()
        gt = masks[i].numpy()
        pr = preds[i].numpy()

        plt.figure(figsize=(12,4))

        plt.subplot(1,3,1)
        plt.title("Image")
        plt.imshow(img)

        plt.subplot(1,3,2)
        plt.title("Ground Truth")
        plt.imshow(gt)

        plt.subplot(1,3,3)
        plt.title("Prediction")
        plt.imshow(pr)

        plt.show()

In [ ]:
def overlay_prediction(image, pred, alpha=0.5):
    plt.imshow(image.permute(1,2,0))
    plt.imshow(pred, alpha=alpha)
    plt.title("Overlay")
    plt.axis('off')
    plt.show()

In [ ]:
def run_evaluation(model, val_loader, device, num_classes):
    
    preds, targets = evaluate_model(model, val_loader, device, num_classes)

    dice_per_class, mean_dice = compute_dice(preds, targets, num_classes)
    iou_per_class, mean_iou = compute_iou(preds, targets, num_classes)
    precision, recall = compute_precision_recall(preds, targets, num_classes)
    accuracy = compute_accuracy(preds, targets)

    print("\n===== METRICS =====")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")

    print("\nPer-class Dice:", dice_per_class)
    print("Per-class IoU:", iou_per_class)
    print("Precision:", precision)
    print("Recall:", recall)

    return preds, targets

In [ ]:


def preprocess_all_folds(dataset, mode_inp):

    folds = []

    for fold_name in ["fold1", "fold2", "fold3"]:
        print(f"\nProcessing {fold_name}")
        print("Samples:", len(dataset[fold_name]))

        imgs, msks = process_fold(
            dataset[fold_name],
            fold_name,
            mode_inp
        )

        folds.append((imgs, msks))

    return folds


def process_fold(hf_dataset, fold_name, mode_inp):

    images, masks = [], []

    for sample in hf_dataset:

        img = preprocess_image(sample["image"])
        msk = preprocess_mask(
            sample["instances"],
            labels=sample["categories"],
            mode=mode_inp
        )

        images.append(img)
        masks.append(msk)

    # Convert to numpy arrays
    images = np.stack(images)
    masks  = np.stack(masks)

    print(f"{fold_name} done:")
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    print("Mask values:", np.unique(masks))

    return images, masks

In [ ]:
def preprocess_image(image):
    #print("\n[IMAGE PREPROCESSING]")
    image = np.array(image)
    #print("Original shape:", image.shape)
    #print("Original dtype:", image.dtype)

    # Normalize to [0,1]
    image = image.astype(np.float32) / 255.0

    # Convert HWC → CHW
    image = np.transpose(image, (2, 0, 1))

    #print("Processed shape (CHW):", image.shape)
    #print("Min/Max values:", image.min(), image.max())

    return image

def preprocess_mask(instances, labels=None, mode="binary", num_classes=6):
    instances = np.array(instances)

   
    # CASE 1: EMPTY (no nuclei)
  
    if instances.size == 0 or instances.ndim == 1:
        if mode == "binary":
            mask = np.zeros((256, 256), dtype=np.uint8)
        else:  # semantic
            mask = np.zeros((256, 256), dtype=np.int64)

    else:
        H, W = instances.shape[1], instances.shape[2]


        # BINARY SEGMENTATION 
        
        if mode == "binary":
            mask = np.any(instances > 0, axis=0)
            mask = mask.astype(np.uint8)

       
        # SEMANTIC SEGMENTATION 
        
        elif mode == "semantic":
            if labels is None:
                raise ValueError("labels required for semantic segmentation")

            mask = np.zeros((H, W), dtype=np.int64)

            for i in range(len(instances)):
                instance_mask = instances[i] > 0   # ensure binary
                class_id = int(labels[i]) + 1      # shift for background=0

                # safer overwrite (handles overlap slightly better)
                mask[instance_mask] = np.maximum(
                    mask[instance_mask],
                    class_id
                )

        else:
            raise ValueError("mode must be 'binary' or 'semantic'")

    
    # SAFETY CHECK
    
    if mask.shape != (256, 256):
        print("\nUnexpected mask shape")
        print("Raw shape:", instances.shape)
        print("After merge:", mask.shape)
        print("Unique:", np.unique(mask))

        if mode == "binary":
            mask = np.zeros((256, 256), dtype=np.uint8)
        else:
            mask = np.zeros((256, 256), dtype=np.int64)


    # CHANNEL HANDLING
   
    if mode == "binary":
        # (1, H, W)
        mask = np.expand_dims(mask, axis=0)

    elif mode == "semantic":
        #CrossEntropyLoss expects (H, W)
        pass

    return mask

In [ ]:
def load_model(model, path):
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    return model


In [ ]:
# =========================
# STEP 0: Preprocess dataset into folds
# =========================
processed_folds = preprocess_all_folds(dataset,"semantic")

print(f"Total folds created: {len(processed_folds)}")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:



# STEP 1: Create DataLoaders for each fold

from torch.utils.data import DataLoader, TensorDataset

test_loaders = []
test_images_list = []

for i in range(3):
    print(f"\n[INFO] Preparing Fold {i}")

    images, masks = processed_folds[i]

    print(f"  Raw samples: {len(images)}")

    images = torch.stack([torch.tensor(x) for x in images])
    masks  = torch.stack([torch.tensor(x) for x in masks])

    print(f"  Image tensor shape: {images.shape}")
    print(f"  Mask tensor shape:  {masks.shape}")

    dataset_fold = TensorDataset(images, masks)

    loader = DataLoader(
        dataset_fold,
        batch_size=8,
        shuffle=False
    )

    print(f"  Batches in loader: {len(loader)}")

    test_loaders.append(loader)
    test_images_list.append(images)



   



In [ ]:

'''
def compute_metrics(preds, targets, num_classes, eps=1e-6):
    """
    preds:   [N, H, W] (after argmax)
    targets: [N, H, W]
    """

    preds = preds.view(-1)
    targets = targets.view(-1)

    dice_per_class = []
    iou_per_class = []
    precision_per_class = []
    recall_per_class = []

    for cls in range(num_classes):
        pred_c = (preds == cls).float()
        target_c = (targets == cls).float()

        TP = (pred_c * target_c).sum()
        FP = (pred_c * (1 - target_c)).sum()
        FN = ((1 - pred_c) * target_c).sum()
        TN = ((1 - pred_c) * (1 - target_c)).sum()

        # Dice
        dice = (2 * TP + eps) / (2 * TP + FP + FN + eps)
        dice_per_class.append(dice.item())

        # IoU
        iou = (TP + eps) / (TP + FP + FN + eps)
        iou_per_class.append(iou.item())

        # Precision
        precision = (TP + eps) / (TP + FP + eps)
        precision_per_class.append(precision.item())

        # Recall
        recall = (TP + eps) / (TP + FN + eps)
        recall_per_class.append(recall.item())

    # Mean metrics (ignore background = class 0 if needed)
    mean_dice = np.mean(dice_per_class[1:])
    mean_iou  = np.mean(iou_per_class[1:])
    mean_precision = np.mean(precision_per_class[1:])
    mean_recall = np.mean(recall_per_class[1:])

    # Overall accuracy
    accuracy = (preds == targets).float().mean().item()

    return {
        "mean_dice": mean_dice,
        "mean_iou": mean_iou,
        "accuracy": accuracy,
        "mean_precision": mean_precision,
        "mean_recall": mean_recall,
        "dice_per_class": dice_per_class,
        "iou_per_class": iou_per_class
    }
'''

In [ ]:
'''
# STEP 2: Run evaluation
num_classes=6
all_results = []
path_lst=[path0,path1,path2]
for i in range(3):
    print(f"\n==============================")
    print(f" EVALUATING FOLD {i}")
    print(f"==============================")

    model = u.unet(6)
    print(f"[INFO] Loading model: best_model_fold{i}.pth")
    
    model = load_model(model, path_lst[i])
    
    

    test_loader = test_loaders[i]
    test_images = test_images_list[i]

    print(f"[INFO] Running inference...")
    preds, targets = run_evaluation(model, test_loader, device, num_classes=6)

    print(f"[INFO] Predictions shape: {preds.shape}")
    print(f"[INFO] Targets shape:     {targets.shape}")

   
    # Metrics
   
    print(f"[INFO] Computing metrics...")
    metrics = compute_metrics(preds, targets, num_classes)

    print("\nMetrics Summary:")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    #  Most important debug insight
    if "dice_per_class" in metrics:
        print(f"\n Per-class Dice:")
        for cls, score in enumerate(metrics["dice_per_class"]):
            print(f"Class {cls}: {score:.4f}")

    if "iou_per_class" in metrics:
        print(f"\n Per-class IoU:")
        for cls, score in enumerate(metrics["iou_per_class"]):
            print(f"Class {cls}: {score:.4f}")
'''

In [ ]:
import gc

In [ ]:
def get_colormap(num_classes):
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(num_classes, 3))
    colors[0] = [0, 0, 0]  # background = black
    return colors
def mask_to_rgb(mask, colormap):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)

    for cls in range(len(colormap)):
        rgb[mask == cls] = colormap[cls]

    return rgb
def overlay_mask(image, mask_rgb, alpha=0.5):
    image = image.astype(np.float32)

    if image.max() <= 1.0:
        image = image * 255

    overlay = image * (1 - alpha) + mask_rgb * alpha
    return overlay.astype(np.uint8)

In [ ]:
def visualize_sample(image, gt_mask, pred_mask, colormap):
    """
    image: [3, H, W]
    gt_mask: [H, W]
    pred_mask: [H, W]
    """

    image = image.permute(1, 2, 0).cpu().numpy()
    gt_mask = gt_mask.cpu().numpy()
    pred_mask = pred_mask.cpu().numpy()

    gt_rgb = mask_to_rgb(gt_mask, colormap)
    pred_rgb = mask_to_rgb(pred_mask, colormap)

    overlay_gt = overlay_mask(image, gt_rgb)
    overlay_pred = overlay_mask(image, pred_rgb)

    fig, axs = plt.subplots(1, 5, figsize=(18, 4))

    axs[0].imshow(image.astype(np.uint8))
    axs[0].set_title("Original")

    axs[1].imshow(gt_rgb)
    axs[1].set_title("GT Mask")

    axs[2].imshow(pred_rgb)
    axs[2].set_title("Prediction")

    axs[3].imshow(overlay_gt)
    axs[3].set_title("GT Overlay")

    axs[4].imshow(overlay_pred)
    axs[4].set_title("Pred Overlay")

    for ax in axs:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
num_classes = 6
path_lst = [path0, path1, path2]

dice_scores = []
iou_scores = []

colormap = get_colormap(num_classes)

for i in range(3):
    print(f"\n==============================")
    print(f" EVALUATING FOLD {i}")
    print(f"==============================")

    model = u.unet(6)
    model = load_model(model, path_lst[i])
    model.to(device)
    model.eval()

    test_loader = test_loaders[i]
    metrics_tracker = StreamingMetrics(num_classes)

    visualized = 0
    MAX_VIS = 5

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(test_loader):

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            metrics_tracker.update(preds, masks)

            
            if visualized < MAX_VIS:
                for j in range(images.shape[0]):
                    visualize_sample(
                        images[j].cpu(),
                        masks[j].cpu(),
                        preds[j].cpu(),
                        colormap
                    )
                    visualized += 1
                    if visualized >= MAX_VIS:
                        break

            # cleanup
            del images, masks, outputs, preds
            if device == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

    metrics = metrics_tracker.compute()

    dice_scores.append(metrics["mean_dice"])
    iou_scores.append(metrics["mean_iou"])

    print("\nMetrics Summary:")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    print("\nPer-class Dice:")
    for cls, score in enumerate(metrics["dice_per_class"]):
        print(f"Class {cls}: {score:.4f}")

    print("\nPer-class IoU:")
    for cls, score in enumerate(metrics["iou_per_class"]):
        print(f"Class {cls}: {score:.4f}")

    del model, metrics_tracker
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


# =========================
# FINAL RESULTS
# =========================
print(f"\n==============================")
print(" FINAL CROSS-VALIDATION RESULTS")
print(f"==============================")

print(f"Mean Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
print(f"Mean IoU:  {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")